# 04 — LLM-as-Judge Evaluation

Run the full RAG pipeline over ground-truth questions with 2 models × 2 prompts.
Measure RELEVANT / PARTLY_RELEVANT / NON_RELEVANT distribution for each combination.

In [1]:
import sys, os, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv(ROOT / '.envrc')

from movie_assistant.rag import rag

# Use gpt-5.4-mini ground truth (switch to gpt-5.6-luna to compare)
GT_MODEL = 'gpt-5.4-mini'
gt = pd.read_csv(ROOT / f'data/ground-truth-retrieval-{GT_MODEL}.csv').sample(200, random_state=42).reset_index(drop=True)
print(f'Evaluating on {len(gt)} questions (ground truth from {GT_MODEL})')
gt.head(3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating on 200 questions (ground truth from gpt-5.4-mini)


,movie_id,question
0,322,Can you recommend a dark thriller about childh...
1,881,What are some good legal dramas involving high...
2,210577,I’m looking for a suspenseful movie where a ma...


In [3]:
MODELS = [
    'gpt-5.6-luna',
    'gpt-5.4-mini',
]
PROMPT_VERSIONS = ['a', 'b']

all_results = []

for model in MODELS:
    for pv in PROMPT_VERSIONS:
        print(f'\n--- model={model}  prompt={pv} ---')
        for _, row in tqdm(gt.iterrows(), total=len(gt)):
            try:
                result = rag(row['question'], model=model, prompt_version=pv)
                all_results.append({
                    'model': model,
                    'prompt_version': pv,
                    'question': row['question'],
                    'movie_id': row['movie_id'],
                    'answer': result.answer,
                    'relevance': result.relevance,
                    'tokens_prompt': result.tokens_prompt,
                    'tokens_completion': result.tokens_completion,
                    'cost': result.cost,
                })
                time.sleep(0.2)
            except Exception as e:
                print(f'  error: {e}')

eval_df = pd.DataFrame(all_results)
eval_df.to_csv(ROOT / 'data/rag-eval-results.csv', index=False)
print(f'\nSaved {len(eval_df)} rows')


--- model=gpt-5.6-luna  prompt=a ---


  0%|          | 0/200 [00:00<?, ?it/s]


--- model=gpt-5.6-luna  prompt=b ---


  0%|          | 0/200 [00:00<?, ?it/s]


--- model=gpt-5.4-mini  prompt=a ---


  0%|          | 0/200 [00:00<?, ?it/s]


--- model=gpt-5.4-mini  prompt=b ---


  0%|          | 0/200 [00:00<?, ?it/s]


Saved 800 rows


In [4]:
# Summary table: RELEVANT % by model × prompt
pivot = (
    eval_df.groupby(['model', 'prompt_version', 'relevance'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
)
for col in ['RELEVANT', 'PARTLY_RELEVANT', 'NON_RELEVANT']:
    if col in pivot:
        pivot[f'{col}_pct'] = (pivot[col] / pivot['total'] * 100).round(1)

pivot[['RELEVANT_pct','PARTLY_RELEVANT_pct','NON_RELEVANT_pct','total']]

relevance                    RELEVANT_pct  PARTLY_RELEVANT_pct  \
model        prompt_version                                      
gpt-5.4-mini a                       69.0                 31.0   
             b                       75.0                 24.5   
gpt-5.6-luna a                       72.0                 26.5   
             b                       77.5                 22.0   

relevance                    NON_RELEVANT_pct  total  
model        prompt_version                           
gpt-5.4-mini a                            0.0    200  
             b                            0.5    200  
gpt-5.6-luna a                            1.5    200  
             b                            0.5    200

In [5]:
# Cost summary
eval_df.groupby(['model','prompt_version'])['cost'].sum().round(4)

model         prompt_version
gpt-5.4-mini  a                 0.1814
              b                 0.1699
gpt-5.6-luna  a                 0.2891
              b                 0.2741
Name: cost, dtype: float64